# Tutorial: Registry Wave 1 Primary-Source Expansion (2026-03-07)

Audience:
- TrustSignal engineering
- Security/compliance reviewers
- Partner auditors (Vanta evidence collection)

Prerequisites:
- Repository checkout at `TrustSignal`
- Node.js 18+
- Access to `apps/api` test database for route-level tests

Learning goals:
- Identify what changed in registry adapter Wave 1.
- Map implementation details to compliance evidence expectations.
- Re-run validation commands for reproducible review.


## Change Scope

This notebook documents the Wave 1 continuation of the free primary-source registry plan:

1. Added primary-source adapters:
   - `openfema_nfip_community` (FEMA OpenFEMA NFIP endpoint)
   - `gleif_lei_records` (GLEIF LEI API)
2. Added parser/provider wiring and fail-closed behavior through existing `COMPLIANCE_GAP` pathways.
3. Expanded API route coverage tests for source listing + verify behavior.
4. Updated planning artifacts (`TASKS.md`, `docs/registry/free_primary_sources_catalog.md`).


In [ ]:
from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path
import subprocess

ROOT = Path.cwd()
ROOT


In [ ]:
changed_files = [
    "apps/api/src/services/registryAdapters.ts",
    "apps/api/src/registry-adapters.test.ts",
    "docs/registry/free_primary_sources_catalog.md",
    "TASKS.md",
    "notebooks/registry-wave1-primary-source-expansion-2026-03-07.ipynb",
]
changed_files


## Security and Trust Guardrails Applied

- Primary-source host validation remains enforced before lookup execution.
- All outbound calls continue using secure request headers and timeout controls.
- Lookup failures continue to resolve to `COMPLIANCE_GAP` (fail-closed), not `PASS`.
- Each verification still emits a registry oracle job commitment for proof linkage.


In [ ]:
vanta_evidence_map = {
    "control.primary_source_integrity": [
        "apps/api/src/services/registryAdapters.ts",
        "docs/registry/free_primary_sources_catalog.md",
    ],
    "control.fail_closed_behavior": [
        "apps/api/src/services/registryAdapters.ts",
        "apps/api/src/registry-adapters.test.ts",
    ],
    "control.change_management_traceability": [
        "TASKS.md",
        "notebooks/registry-wave1-primary-source-expansion-2026-03-07.ipynb",
    ],
}
vanta_evidence_map


## Validation Commands

Run the following commands from repository root to reproduce verification evidence:


In [ ]:
commands = [
    "npm -w apps/api run test -- src/registry-adapters.test.ts",
    "npm -w apps/api run typecheck",
    "git status --short",
]

for cmd in commands:
    print(cmd)


## TrustSignal History Snapshot (for audit context)

This section captures recent repository evolution so reviewers can place the registry changes in context.


In [ ]:
history_cmd = [
    "git",
    "log",
    "--date=short",
    "--pretty=format:%h|%ad|%s",
    "--max-count=30",
]
history_output = subprocess.run(history_cmd, capture_output=True, text=True, check=True).stdout.splitlines()
history_output[:15]


### Milestone summary from recent history

- `2026-03-06` `1182ad6`/`e6b1d1e`: MVP10 registry baseline landed.
- `2026-03-06` `00264c3`/`227a55c`: primary-source guardrails and fail-closed registry artifacts hardened.
- `2026-03-05` `db288a2`/`37cb532`: Vanta integration outputs and partner demo package added.
- `2026-03-02` `7466da4` and session commits: API, SDK, testing, and threat-model posture matured.
- `2026-02-27` `9930280`: dependency security patching committed.

Wave 1 expansion (`openfema_nfip_community`, `gleif_lei_records`) extends this same primary-source + fail-closed pattern.


## 90-Day Commit Timeline

This cell derives a rolling 90-day history window from the current date so evidence exports stay current.


In [ ]:
since_date = (date.today() - timedelta(days=90)).isoformat()
timeline_cmd = [
    "git",
    "log",
    f"--since={since_date}",
    "--date=short",
    "--pretty=format:%h|%ad|%s",
]
timeline_output = subprocess.run(timeline_cmd, capture_output=True, text=True, check=True).stdout.splitlines()
len(timeline_output), timeline_output[:20]


In [ ]:
area_keywords = {
    "registry": ["registry", "registries", "primary-source"],
    "security": ["security", "audit", "harden", "secret", "tls"],
    "vanta": ["vanta", "soc2", "compliance"],
    "api": ["api", "fastify", "route", "sdk"],
}

summary = {key: 0 for key in area_keywords}
summary["other"] = 0

for line in timeline_output:
    lower = line.lower()
    matched = False
    for area, keywords in area_keywords.items():
        if any(keyword in lower for keyword in keywords):
            summary[area] += 1
            matched = True
            break
    if not matched:
        summary["other"] += 1

summary


## Vanta Control Mapping (Detailed)

The structure below is export-friendly and ties controls directly to in-repo artifacts and validation commands.


In [ ]:
vanta_control_export_rows = [
    {
        "control_id": "CC6.1",
        "control_name": "Logical Access and Authentication",
        "objective": "Enforce authenticated and scoped access to API routes.",
        "evidence_files": [
            "apps/api/src/security.ts",
            "apps/api/src/server.ts",
        ],
        "validation_commands": [
            "npm -w apps/api run typecheck",
            "npm -w apps/api run test -- src/registry-adapters.test.ts",
        ],
    },
    {
        "control_id": "CC7.2",
        "control_name": "Change Management and Risk Mitigation",
        "objective": "Track planned registry rollout and implementation status.",
        "evidence_files": [
            "TASKS.md",
            "docs/registry/free_primary_sources_catalog.md",
            "notebooks/registry-wave1-primary-source-expansion-2026-03-07.ipynb",
        ],
        "validation_commands": [
            "git log --date=short --pretty=format:'%h|%ad|%s' --max-count=30",
            "git status --short",
        ],
    },
    {
        "control_id": "CC7.3",
        "control_name": "Security Event and Failure Handling",
        "objective": "Fail closed on upstream registry unavailability and preserve evidence.",
        "evidence_files": [
            "apps/api/src/services/registryAdapters.ts",
            "apps/api/src/registry-adapters.test.ts",
            "SECURITY_CHECKLIST.md",
        ],
        "validation_commands": [
            "npm -w apps/api run test -- src/registry-adapters.test.ts",
        ],
    },
    {
        "control_id": "CC8.1",
        "control_name": "Vendor and External Dependency Governance",
        "objective": "Use only official primary-source registry endpoints.",
        "evidence_files": [
            "apps/api/src/services/registryAdapters.ts",
            "docs/registry/free_primary_sources_catalog.md",
        ],
        "validation_commands": [
            "rg -n 'primarySourceHost|officialSourceName' apps/api/src/services/registryAdapters.ts",
        ],
    },
    {
        "control_id": "CC9.2",
        "control_name": "Operational Monitoring and Incident Readiness",
        "objective": "Maintain operational readiness and governance evidence continuity.",
        "evidence_files": [
            "docs/final/10_INCIDENT_ESCALATION_AND_SLO_BASELINE.md",
            "docs/PRODUCTION_GOVERNANCE_TRACKER.md",
            "docs/final/13_SOC2_READINESS_KICKOFF.md",
        ],
        "validation_commands": [
            "rg -n 'status|SLO|incident|governance' docs/final/10_INCIDENT_ESCALATION_AND_SLO_BASELINE.md docs/PRODUCTION_GOVERNANCE_TRACKER.md",
        ],
    },
]

vanta_control_export_rows


In [ ]:
import csv

export_path = ROOT / "notebooks" / "artifacts" / "vanta-controls-registry-wave1-2026-03-07.csv"
export_path.parent.mkdir(parents=True, exist_ok=True)

with export_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["control_id", "control_name", "objective", "evidence_files", "validation_commands"])
    writer.writeheader()
    for row in vanta_control_export_rows:
        writer.writerow({
            "control_id": row["control_id"],
            "control_name": row["control_name"],
            "objective": row["objective"],
            "evidence_files": " | ".join(row["evidence_files"]),
            "validation_commands": " | ".join(row["validation_commands"]),
        })

str(export_path)


## Review Sign-off Checklist

- [ ] Adapter IDs listed in `/api/v1/registry/sources`
- [ ] Positive match path verified for new providers
- [ ] `COMPLIANCE_GAP` behavior unchanged for unavailable providers
- [ ] Task tracker and expansion catalog updated
- [ ] Notebook retained as audit evidence artifact
